In [ ]:
# import knihoven
import pandas as pd
import numpy as np
from uncertainties import ufloat as uf
from uncertainties import unumpy as unp
from pathlib import Path
import matplotlib.pyplot as plt

# path config
downloads_dir = Path.home() / "Downloads"
# ujistime se, ze slozka existuje (kdyby nahodou ne)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'muj_graf.png'

In [ ]:
# definice funkci

# nejistota typu B u cteni z nejmensiho dilku
def unc_B_cteni(a):
    """
    Nejistota typu B pro napr pravitko
    data : float
        nejmensi dilek
    return : float
        nejistota typu B
    """
    u_B = a/(3**0.5)
    return u_B

# nejistota typu A 
def unc_A(data):
    """
    Výpočet nejistoty typu A
    data : list nebo numpy array
         měření
    return : float
        nejistota typu A
    """
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)
    
    return u_A

In [ ]:
# nacteni google tabulek
sheet_id = '1X6DslFJgiO2eAeN-leIRMBTIs06rjy_FhxOTD-FuwGI'
gid = '0'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_cocky = pd.read_csv(url)
df_cocky.dropna()

In [ ]:
# cocky

cteni_unc = unc_B_cteni(0.2) # v cm

spojka, stinitko, zdroj, sipka, delta, predmet, rozptylka, obraz, pocatek = (unp.uarray(df_cocky[c].dropna().to_numpy(), cteni_unc)*1e-2 for c in df_cocky.columns)
pocatek = pocatek[0]
zdroj = zdroj[0]
veliciny = [spojka, stinitko, delta, predmet, rozptylka, obraz]
veliciny = [v - pocatek for v in veliciny]
[spojka, stinitko, delta, predmet, rozptylka, obraz] = veliciny
predmet = predmet[0]
spojka = -spojka
stinitko = stinitko + spojka
stinitko_povod = stinitko - spojka
f_prima = (spojka * stinitko)/(spojka - stinitko)
#for u in f_prima: print(fr'${u*100:.1uL}$') # na centimetry

zvetseni = -sipka/zdroj
f_zvetseni = (spojka*zvetseni)/(1-zvetseni)
#for u in f_zvetseni:print(fr'${u*100:.1uL}$') # na centimetry

delta_true =  delta + spojka
f_bessel = ((stinitko_povod)**2 - delta_true**2)/(4*stinitko_povod)
#for u in f_bessel:print(fr'${u*100:.1uL}$') # na centimetry

f_s_P = np.mean(unp.nominal_values(f_bessel))

predmet_roz = predmet - rozptylka
stinitko_roz = obraz - rozptylka
f_rozptylka = (predmet_roz*stinitko_roz)/(predmet_roz-stinitko_roz)
#for u in f_rozptylka:print(fr'${u*100:.1uL}$') # na centimetry

f_r_P = np.mean(unp.nominal_values(f_rozptylka))
f_r_P = f_rozptylka[8]
#for u in unp.nominal_values(f_prima):print(fr'${u*100:.1f}$')
f_r_P

-0.26904090909090905+/-0.004438368962849261

In [ ]:
fig, ax1 = plt.subplots(figsize=(16, 9))
ax1.set_ylabel(fr"$f-spojka\,$[cm]", fontsize=20, color="tab:blue")
ax1.set_xlabel(fr"vzdálenost stínítka$\,$[cm]", fontsize=20)
ax1.scatter(unp.nominal_values(stinitko_povod)*100, unp.nominal_values(f_prima)*100, label='Přímá metoda - spojka', color='blue', s=100)
ax1.plot(unp.nominal_values(stinitko_povod)*100, unp.nominal_values(f_prima)*100, color='blue')
ax1.tick_params(axis="y", labelsize=15, labelcolor="tab:blue")
ax1.tick_params(axis="x", labelsize=15)
#ax1.grid(True, alpha=0.7)
ax1.legend(fontsize=15,loc="upper center")

ax2 = ax1.twinx()
ax2.scatter(unp.nominal_values(stinitko_povod)*100, unp.nominal_values(f_rozptylka)*100, label='Přímá metoda - rozptylka', color='red', s=100)
ax2.plot(unp.nominal_values(stinitko_povod)*100, unp.nominal_values(f_rozptylka)*100, color='red')
ax2.set_ylabel(fr"$f-rozptylka\,$[cm]", fontsize=20, color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red",labelsize=15)
ax2.legend(fontsize=15,loc="upper right")
path_chyba = downloads_dir / 'ohnisko_chyba.png'
plt.savefig(path_chyba, dpi=300, bbox_inches='tight')

In [ ]:
# nacteni google tabulek
sheet_id = '1X6DslFJgiO2eAeN-leIRMBTIs06rjy_FhxOTD-FuwGI'
gid = '708172543'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_sfero = pd.read_csv(url)
df_sfero.dropna()

In [30]:
# sferometr
h_s1, z_s, h_s2, h_r1, z_r, h_r2= (df_sfero[c].dropna().to_numpy()*1e-3 for c in df_sfero.columns)
h_s1 = uf(np.mean(h_s1), (unc_A(h_s1)**2 + (0.000001)**2)**0.5)
z_s = uf(np.mean(z_s), (unc_A(z_s)**2 + (0.00002)**2)**0.5)/2
h_s2 = uf(np.mean(h_s2), (unc_A(h_s2)**2 + (0.000001)**2)**0.5)

h_r1 = uf(-np.mean(h_r1), (unc_A(h_r1)**2 + (0.000001)**2)**0.5)
z_r = uf(np.mean(z_r), (unc_A(z_r)**2 + (0.00002)**2)**0.5)/2
h_r2 = uf(-np.mean(h_r2), (unc_A(h_r2)**2 + (0.000001)**2)**0.5)


r_s1 = (z_s**2 + h_s1**2)/(2*h_s1)
r_s2 = -(z_s**2 + h_s2**2)/(2*h_s2)

r_r1 = (z_r**2 + h_r1**2)/(2*h_r1)
r_r2 = -(z_r**2 + h_r2**2)/(2*h_r2)

print(f'${r_s1*100:.1uL}$')
print(f'${r_s2*100:.1uL}$')
print()
print(f'${r_r1*100:.1uL}$')
print(f'${r_r2*100:.1uL}$')

$\left(3.6 \pm 0.9\right) \times 10^{3}$
$-8.42 \pm 0.01$

$-34.17 \pm 0.08$
$34.00 \pm 0.08$


In [ ]:
n_sp = 1 + (1/f_s_P)/(1/r_s1 - 1/(r_s2))
print(n_sp)

n_rp = 1 + (1/f_r_P)/(1/r_r1 - 1/(r_r2))
print(n_rp)